### Transformers ([Attention Is All You Need](https://arxiv.org/pdf/1706.03762.pdf))

RNNs are effective for sequence-to-sequence tasks but have two important limitations. Long sequences can cause vanishing gradients, and sequential dependencies between hidden states limit parallel training on modern GPUs.

LSTMs and GRUs address much of the gradient problem, but they retain sequential computation. Transformers reduce this limitation by processing sequence elements in parallel during training. The attention computation has quadratic complexity with respect to sequence length, which remains practical for many modern GPU workloads.

This notebook develops a Transformer model step by step using the original [Attention Is All You Need](https://arxiv.org/pdf/1706.03762.pdf) paper as a reference. A toy vector-to-vector dataset demonstrates a simplified sequence-to-sequence task.

## Table of Contents

The notebook contains four parts. The focus is an encoder-based Transformer, while full sequence-to-sequence systems commonly combine an encoder and a decoder.

1. **Part I: Transformer building blocks**
   1. Multi-Head Attention
   2. Feed-Forward Network
   3. Layer Normalization
   4. Encoder Block
2. **Part II: Data preparation**
3. **Part III: Model training**
4. **Part IV: Sentiment analysis evaluation**

CPU execution is sufficient for the initial sections. The later training section benefits from GPU acceleration and depends on the preceding components.

![Encoder Block](https://miro.medium.com/v2/resize:fit:880/format:webp/1*Wew2tXiDk_rMPFCq73cysw.png)

In [2]:
%load_ext autoreload
%autoreload 2

### Google Colab Setup

This section contains the commands required for a Google Colab environment. It can be skipped when the notebook runs on a local machine.

The next code cell mounts Google Drive. The authorization code is entered in the prompt after authentication with the Google account associated with the notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import sys

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = None

GOOGLE_DRIVE_PATH = os.path.join("/content/drive/MyDrive/Transformer")
print(os.listdir(GOOGLE_DRIVE_PATH))

# Add to sys so we can import .py files.

sys.path.append(GOOGLE_DRIVE_PATH)

['a5_helper.py', '.DS_Store', '__pycache__', 'helpers_module', 'transformers_sentiment_analysis.py', 'transformers_sentiment_analysis.ipynb']


After Google Drive is mounted and the path is configured, the next code cell makes the `.py` source files available for import. Successful execution prints:

```text
Hello from transformers_sentiment_analysis.py!
```

The output also includes the last edit time of `transformers_sentiment_analysis.py`.

In [4]:
import os
import time
from transformers_sentiment_analysis import hello_transformers


os.environ["TZ"] = "Asia/Tehran"
time.tzset()
hello_transformers()

transformers_path = os.path.join(GOOGLE_DRIVE_PATH, "transformers_sentiment_analysis.py")
transformers_edit_time = time.ctime(os.path.getmtime(transformers_path))
print("transformers.py last edited on %s" % transformers_edit_time)

Hello from transformers_sentiment_analysis.py!
transformers.py last edited on Mon Dec 18 00:06:22 2023


In [5]:
import torch
import torch.nn.functional as F
from torch import Tensor
from torch import nn

import torch

from torch import nn
import torch.nn.functional as F

from helpers_module.utils import (
    reset_seed,
    tensor_to_image,
    attention_visualizer,
)
from helpers_module.grad import rel_error, compute_numeric_gradient
import matplotlib.pyplot as plt
import time
from IPython.display import Image


# for plotting
%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0)  # set default size of plots
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

GPU acceleration is used for the computationally intensive sections. The code selects the available device and uses `torch.float32` for data tensors and `torch.int64` (`torch.long`) for labels.

Additional details are available in the [PyTorch tensor attributes documentation](https://pytorch.org/docs/stable/tensor_attributes.html#torch-dtype).

In [6]:
to_float = torch.float
to_long = torch.long

if torch.cuda.is_available():
    print("Good to go!")
    DEVICE = torch.device("cuda")
else:
    print("Please set GPU via Edit -> Notebook Settings.")
    DEVICE = torch.device("cpu")

Please set GPU via Edit -> Notebook Settings.


# Part I: Transformer Building Blocks

The first part introduces sentiment analysis and the building blocks of a Transformer model. Each component is implemented as a subclass of `nn.Module`, with PyTorch autograd handling gradient computation.

The main components are:

1. Multi-Head Attention
2. Feed-Forward Network
3. Layer Normalization
4. Positional Encoding

These components, together with an input embedding layer, form the Transformer Encoder. Tensor shape consistency between each block's input and output is an important implementation check.

### MultiHeadAttention Block

![Encoder Block](https://uvadlc-notebooks.readthedocs.io/en/latest/_images/multihead_attention.svg)

Transformers map input sequences to output sequences. Input tokens are first converted into embeddings and then combined with positional encodings. Learnable projections produce query, key, and value tensors, which are processed by attention and feed-forward components.

The encoder places a multi-head attention block after the input representation. Decoder self-attention is masked in sequence-to-sequence architectures but is outside the scope of this implementation.

## Self-Attention Block

The query-key-value formulation comes from information retrieval. Each query retrieves information from key-value pairs by computing dot products with the keys, applying softmax to obtain weights, and forming a weighted sum of the values.

The code in `transformers.py` defines three scaled dot-product attention functions: `scaled_dot_product_two_loop_single`, `scaled_dot_product_two_loop_batch`, and `scaled_dot_product_no_loop_batch`. Numerical checks are expected to produce errors below `1e-5`.

In [7]:
from transformers_sentiment_analysis import (
    scaled_dot_product_two_loop_single,
    scaled_dot_product_two_loop_batch,
    scaled_dot_product_no_loop_batch,
)

In [8]:
N = 2  # Number of sentences
K = 5  # Number of words in a sentence
M = 4  # feature dimension of each word embedding

query = torch.linspace(-0.4, 0.6, steps=K * M).reshape(K, M)  # **to_double_cuda
key = torch.linspace(-0.8, 0.5, steps=K * M).reshape(K, M)  # **to_double_cuda
value = torch.linspace(-0.3, 0.8, steps=K * M).reshape(K, M)  # *to_double_cuda

y = scaled_dot_product_two_loop_single(query, key, value)
y_expected = torch.tensor(
    [
        [0.08283, 0.14073, 0.19862, 0.25652],
        [0.13518, 0.19308, 0.25097, 0.30887],
        [0.18848, 0.24637, 0.30427, 0.36216],
        [0.24091, 0.29881, 0.35670, 0.41460],
        [0.29081, 0.34871, 0.40660, 0.46450],
    ]
).to(torch.float32)
print("sacled_dot_product_two_loop_single error: ", rel_error(y_expected, y))

sacled_dot_product_two_loop_single error:  5.196977309676265e-06


In [9]:
N = 2  # Number of sentences
K = 5  # Number of words in a sentence
M = 4  # feature dimension of each word embedding

query = torch.linspace(-0.4, 0.6, steps=N * K * M).reshape(N, K, M)  # **to_double_cuda
key = torch.linspace(-0.8, 0.5, steps=N * K * M).reshape(N, K, M)  # **to_double_cuda
value = torch.linspace(-0.3, 0.8, steps=N * K * M).reshape(N, K, M)  # *to_double_cuda

y = scaled_dot_product_two_loop_batch(query, key, value)
y_expected = torch.tensor(
    [
        [
            [-0.09603, -0.06782, -0.03962, -0.01141],
            [-0.08991, -0.06170, -0.03350, -0.00529],
            [-0.08376, -0.05556, -0.02735, 0.00085],
            [-0.07760, -0.04939, -0.02119, 0.00702],
            [-0.07143, -0.04322, -0.01502, 0.01319],
        ],
        [
            [0.49884, 0.52705, 0.55525, 0.58346],
            [0.50499, 0.53319, 0.56140, 0.58960],
            [0.51111, 0.53931, 0.56752, 0.59572],
            [0.51718, 0.54539, 0.57359, 0.60180],
            [0.52321, 0.55141, 0.57962, 0.60782],
        ],
    ]
).to(torch.float32)
print("scaled_dot_product_two_loop_batch error: ", rel_error(y_expected, y))

scaled_dot_product_two_loop_batch error:  4.069603357824827e-06


In [10]:
N = 2  # Number of sentences
K = 5  # Number of words in a sentence
M = 4  # feature dimension of each word embedding

query = torch.linspace(-0.4, 0.6, steps=N * K * M).reshape(N, K, M)  # **to_double_cuda
key = torch.linspace(-0.8, 0.5, steps=N * K * M).reshape(N, K, M)  # **to_double_cuda
value = torch.linspace(-0.3, 0.8, steps=N * K * M).reshape(N, K, M)  # *to_double_cuda


y, _ = scaled_dot_product_no_loop_batch(query, key, value)

y_expected = torch.tensor(
    [
        [
            [-0.09603, -0.06782, -0.03962, -0.01141],
            [-0.08991, -0.06170, -0.03350, -0.00529],
            [-0.08376, -0.05556, -0.02735, 0.00085],
            [-0.07760, -0.04939, -0.02119, 0.00702],
            [-0.07143, -0.04322, -0.01502, 0.01319],
        ],
        [
            [0.49884, 0.52705, 0.55525, 0.58346],
            [0.50499, 0.53319, 0.56140, 0.58960],
            [0.51111, 0.53931, 0.56752, 0.59572],
            [0.51718, 0.54539, 0.57359, 0.60180],
            [0.52321, 0.55141, 0.57962, 0.60782],
        ],
    ]
).to(torch.float32)

print("scaled_dot_product_no_loop_batch error: ", rel_error(y_expected, y))

scaled_dot_product_no_loop_batch error:  4.020571992067902e-06


## Observing Time Complexity

Self-attention complexity depends on the input sequence length. The following cells compare sequence lengths of 256 and 512 using `self_attention_no_loop`.

Doubling the sequence length increases the quadratic attention component by approximately four times. The `%timeit` measurements may require several seconds to complete.

In [ ]:
N = 64
K = 256  # defines the input sequence length
M = emb_size = 2048
dim_q = dim_k = 2048
query = torch.linspace(-0.4, 0.6, steps=N * K * M).reshape(N, K, M)  # **to_double_cuda
key = torch.linspace(-0.8, 0.5, steps=N * K * M).reshape(N, K, M)  # **to_double_cuda
value = torch.linspace(-0.3, 0.8, steps=N * K * M).reshape(N, K, M)  # *to_double_cuda

%timeit -n 5 -r 2  y = scaled_dot_product_no_loop_batch(query, key, value)

465 ms ± 22.4 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


In [ ]:
N = 64
K = 512  # defines the input requence length
M = emb_size = 2048
dim_q = dim_k = 2048
query = torch.linspace(-0.4, 0.6, steps=N * K * M).reshape(N, K, M)  # **to_double_cuda
key = torch.linspace(-0.8, 0.5, steps=N * K * M).reshape(N, K, M)  # **to_double_cuda
value = torch.linspace(-0.3, 0.8, steps=N * K * M).reshape(N, K, M)  # *to_double_cuda

%timeit -n 5 -r 2  y = scaled_dot_product_no_loop_batch(query, key, value)

1.63 s ± 176 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


The vectorized scaled dot-product implementation provides the foundation for a single-head attention module. `SingleHeadAttention` inherits from `nn.Module`, with its `__init__` and `forward` methods defined in `transformers.py`.

The following cells evaluate the `SelfAttention` layer and compare its gradients with PyTorch autograd. Expected forward and backward errors are below `1e-5`.

In [12]:
from transformers_sentiment_analysis import SelfAttention

In [13]:
reset_seed(0)
N = 2
K = 4
M = emb_size = 4
dim_q = dim_k = 4
atten_single = SelfAttention(emb_size, dim_q, dim_k)

for k, v in atten_single.named_parameters():
    # print(k, v.shape) # uncomment this to see the weight shape
    v.data.copy_(torch.linspace(-1.4, 1.3, steps=v.numel()).reshape(*v.shape))

query = torch.linspace(-0.4, 0.6, steps=N * K * M, requires_grad=True).reshape(
    N, K, M
)  # **to_double_cuda
key = torch.linspace(-0.8, 0.5, steps=N * K * M, requires_grad=True).reshape(
    N, K, M
)  # **to_double_cuda
value = torch.linspace(-0.3, 0.8, steps=N * K * M, requires_grad=True).reshape(
    N, K, M
)  # *to_double_cuda

query.retain_grad()
key.retain_grad()
value.retain_grad()

y_expected = torch.tensor(
    [
        [
            [-1.10382, -0.37219, 0.35944, 1.09108],
            [-1.45792, -0.50067, 0.45658, 1.41384],
            [-1.74349, -0.60428, 0.53493, 1.67414],
            [-1.92584, -0.67044, 0.58495, 1.84035],
        ],
        [
            [-4.59671, -1.63952, 1.31767, 4.27486],
            [-4.65586, -1.66098, 1.33390, 4.32877],
            [-4.69005, -1.67339, 1.34328, 4.35994],
            [-4.71039, -1.68077, 1.34886, 4.37848],
        ],
    ]
)

dy_expected = torch.tensor(
    [
        [
            [-0.09084, -0.08961, -0.08838, -0.08715],
            [0.69305, 0.68366, 0.67426, 0.66487],
            [-0.88989, -0.87783, -0.86576, -0.85370],
            [0.25859, 0.25509, 0.25158, 0.24808],
        ],
        [
            [-0.05360, -0.05287, -0.05214, -0.05142],
            [0.11627, 0.11470, 0.11312, 0.11154],
            [-0.01048, -0.01034, -0.01019, -0.01005],
            [-0.03908, -0.03855, -0.03802, -0.03749],
        ],
    ]
)

y = atten_single(query, key, value)
dy = torch.randn(*y.shape)  # , **to_double_cuda

y.backward(dy)
query_grad = query.grad

print("SelfAttention error: ", rel_error(y_expected, y))
print("SelfAttention error: ", rel_error(dy_expected, query_grad))

SelfAttention error:  5.282987963847609e-07
SelfAttention error:  2.474069076879365e-06


The single-head attention block provides the basis for multi-head attention. Multiple attention heads process the same input, and their outputs are concatenated along the feature dimension. This simplified implementation uses multiple `SingleHeadAttention` modules for clarity rather than the parameter layout used in production implementations.

The `MultiHeadAttention` block in `transformers.py` combines these single-head components.

The following cells evaluate the `MultiHeadAttention` layer and compare its gradients with PyTorch autograd. Expected error values are below `1e-5`.

In [14]:
from transformers_sentiment_analysis import MultiHeadAttention

In [15]:
reset_seed(0)
N = 2
num_heads = 2
K = 4
M = inp_emb_size = 4
out_emb_size = 8
atten_multihead = MultiHeadAttention(num_heads, inp_emb_size, out_emb_size)

for k, v in atten_multihead.named_parameters():
    # print(k, v.shape) # uncomment this to see the weight shape
    v.data.copy_(torch.linspace(-1.4, 1.3, steps=v.numel()).reshape(*v.shape))

query = torch.linspace(-0.4, 0.6, steps=N * K * M, requires_grad=True).reshape(
    N, K, M
)  # **to_double_cuda
key = torch.linspace(-0.8, 0.5, steps=N * K * M, requires_grad=True).reshape(
    N, K, M
)  # **to_double_cuda
value = torch.linspace(-0.3, 0.8, steps=N * K * M, requires_grad=True).reshape(
    N, K, M
)  # *to_double_cuda

query.retain_grad()
key.retain_grad()
value.retain_grad()

y_expected = torch.tensor(
    [
        [
            [-0.23104, 0.50132, 1.23367, 1.96603],
            [0.68324, 1.17869, 1.67413, 2.16958],
            [1.40236, 1.71147, 2.02058, 2.32969],
            [1.77330, 1.98629, 2.19928, 2.41227],
        ],
        [
            [6.74946, 5.67302, 4.59659, 3.52015],
            [6.82813, 5.73131, 4.63449, 3.53767],
            [6.86686, 5.76001, 4.65315, 3.54630],
            [6.88665, 5.77466, 4.66268, 3.55070],
        ],
    ]
)
dy_expected = torch.tensor(
    [[[ 0.56268,  0.55889,  0.55510,  0.55131],
         [ 0.43286,  0.42994,  0.42702,  0.42411],
         [ 2.29865,  2.28316,  2.26767,  2.25218],
         [ 0.49172,  0.48841,  0.48509,  0.48178]],

        [[ 0.25083,  0.24914,  0.24745,  0.24576],
         [ 0.14949,  0.14849,  0.14748,  0.14647],
         [-0.03105, -0.03084, -0.03063, -0.03043],
         [-0.02082, -0.02068, -0.02054, -0.02040]]]
)

y = atten_multihead(query, key, value)
dy = torch.randn(*y.shape)  # , **to_double_cuda

y.backward(dy)
query_grad = query.grad
print("MultiHeadAttention error: ", rel_error(y_expected, y))
print("MultiHeadAttention error: ", rel_error(dy_expected, query_grad))

MultiHeadAttention error:  5.366163452092416e-07
MultiHeadAttention error:  1.0


### LayerNormalization

Batch normalization depends on statistics from the complete batch, which can be less effective for small batch sizes. Layer normalization avoids this dependency by normalizing each sequence element independently. It is well suited to sequence-to-sequence models and supports parallel computation across sequence positions.

The `LayerNormalization` class defines the forward computation, while PyTorch autograd handles the backward pass. The numerical check is expected to produce an error below `1e-5`.

In [ ]:
from transformers_sentiment_analysis import LayerNormalization

In [ ]:
reset_seed(0)
N = 2
K = 4
norm = LayerNormalization(K)
inp = torch.linspace(-0.4, 0.6, steps=N * K, requires_grad=True).reshape(N, K)

inp.retain_grad()
y = norm(inp)

y_expected = torch.tensor(
    [[-1.34164, -0.44721, 0.44721, 1.34164], [-1.34164, -0.44721, 0.44721, 1.34164]]
)

dy_expected = torch.tensor(
    [[  5.70524,  -2.77289, -11.56993,   8.63758],
        [  2.26242,  -4.44330,   2.09933,   0.08154]]
)

dy = torch.randn(*y.shape)
y.backward(dy)
inp_grad = inp.grad

print("LayerNormalization error: ", rel_error(y_expected, y))
print("LayerNormalization grad error: ", rel_error(dy_expected, inp_grad))

LayerNormalization error:  1.3772273765080196e-06
LayerNormalization grad error:  2.0542348921649034e-07


### FeedForward Block

In the image below we have highlighted the parts where FeedForward Block is used.
<img src="https://drive.google.com/uc?export=view&id=1WCNACnI-Q6OfU3ngjIMCbNzb1sbFnCgP" alt="Layer_norm" width="80%">

The `FeedForward` block appears in both the Encoder and Decoder. It consists of stacked multilayer perceptron and ReLU layers, with the output of `MultiHeadAttention` passed into the feed-forward transformation.

The `FeedForwardBlock` implementation is defined in `transformers.py`. The following cells evaluate its forward output and gradient behavior, with expected errors below `1e-5`.

In [ ]:
from transformers_sentiment_analysis import FeedForwardBlock

In [ ]:
reset_seed(0)
N = 2
K = 4
M = emb_size = 4

ff_block = FeedForwardBlock(emb_size, 2 * emb_size)

for k, v in ff_block.named_parameters():
    v.data.copy_(torch.linspace(-1.4, 1.3, steps=v.numel()).reshape(*v.shape))

inp = torch.linspace(-0.4, 0.6, steps=N * K, requires_grad=True).reshape(
    N, K
)
inp.retain_grad()
y = ff_block(inp)

y_expected = torch.tensor(
    [[-2.46161, -0.71662, 1.02838, 2.77337], [-7.56084, -1.69557, 4.16970, 10.03497]]
)

dy_expected = torch.tensor(
    [[0.55105, 0.68884, 0.82662, 0.96441], [0.30734, 0.31821, 0.32908, 0.33996]]
)

dy = torch.randn(*y.shape)
y.backward(dy)
inp_grad = inp.grad

print("FeedForwardBlock error: ", rel_error(y_expected, y))
print("FeedForwardBlock error: ", rel_error(dy_expected, inp_grad))

FeedForwardBlock error:  2.1976866936034156e-07
FeedForwardBlock error:  1.0


The remaining Transformer encoder components are:

- Encapsulation of the building blocks into an Encoder Block
- Input preprocessing and positional encoding

Positional encoding is a non-learnable embedding and can be applied during data loading as a preprocessing step.

The encoder block combines the previously defined attention, normalization, and feed-forward components. Residual connections connect the outputs of the main sublayers.

The encoder block receives query, key, and value tensors. The following code evaluates the `EncoderBlock`; expected errors are below `1e-5`.

In [ ]:
from transformers_sentiment_analysis import EncoderBlock

In [ ]:
reset_seed(0)
N = 2
num_heads = 2
emb_dim = K = 4
feedforward_dim = 8
M = inp_emb_size = 4
out_emb_size = 8
dropout = 0.2

enc_seq_inp = torch.linspace(-0.4, 0.6, steps=N * K * M, requires_grad=True).reshape(
    N, K, M
)  # **to_double_cuda

enc_block = EncoderBlock(num_heads, emb_dim, feedforward_dim, dropout)

for k, v in enc_block.named_parameters():
    # print(k, v.shape) # uncomment this to see the weight shape
    v.data.copy_(torch.linspace(-1.4, 1.3, steps=v.numel()).reshape(*v.shape))

encoder_out1_expected = torch.tensor(
    [[[ 0.00000, -0.31357,  0.69126,  0.00000],
         [ 0.42630, -0.25859,  0.72412,  3.87013],
         [ 0.00000, -0.31357,  0.69126,  3.89884],
         [ 0.47986, -0.30568,  0.69082,  3.90563]],

        [[ 0.00000, -0.31641,  0.69000,  3.89921],
         [ 0.47986, -0.30568,  0.69082,  3.90563],
         [ 0.47986, -0.30568,  0.69082,  3.90563],
         [ 0.51781, -0.30853,  0.71598,  3.85171]]]
)
encoder_out1 = enc_block(enc_seq_inp)
print("EncoderBlock error 1: ", rel_error(encoder_out1, encoder_out1_expected))


N = 2
num_heads = 1
emb_dim = K = 4
feedforward_dim = 8
M = inp_emb_size = 4
out_emb_size = 8
dropout = 0.2

enc_seq_inp = torch.linspace(-0.4, 0.6, steps=N * K * M, requires_grad=True).reshape(
    N, K, M
)  # **to_double_cuda

enc_block = EncoderBlock(num_heads, emb_dim, feedforward_dim, dropout)

for k, v in enc_block.named_parameters():
    # print(k, v.shape) # uncomment this to see the weight shape
    v.data.copy_(torch.linspace(-1.4, 1.3, steps=v.numel()).reshape(*v.shape))

encoder_out2_expected = torch.tensor(
    [[[ 0.42630, -0.00000,  0.72412,  3.87013],
         [ 0.49614, -0.31357,  0.00000,  3.89884],
         [ 0.47986, -0.30568,  0.69082,  0.00000],
         [ 0.51654, -0.32455,  0.69035,  3.89216]],

        [[ 0.47986, -0.30568,  0.69082,  0.00000],
         [ 0.49614, -0.31357,  0.69126,  3.89884],
         [ 0.00000, -0.30354,  0.76272,  3.75311],
         [ 0.49614, -0.31357,  0.69126,  3.89884]]]
)
encoder_out2 = enc_block(enc_seq_inp)
print("EncoderBlock error 2: ", rel_error(encoder_out2, encoder_out2_expected))

EncoderBlock error 1:  0.4938230406847835
EncoderBlock error 2:  0.5710266743125565


The Transformer building blocks are now assembled into the main model components.

The decoder block completes the remaining Transformer component. Its `__init__` and `forward` methods are defined in `transformers.py`, and the following code evaluates the `DecoderBlock` with expected errors below `1e-5`.

The `Encoder` and `Decoder` networks are provided in `transformers.py`. Their input and output interfaces form the basis for the complete Transformer module.

## Part III: Data Loader

This section prepares the final data loader for Transformer training. The main tasks are positional encoding and construction of a data loader using the `preprocess_input_sequence` function.

Positional encodings provide sequence-order information to the Transformer. They are added to the input embeddings, have the same shape as the input, and remain constant during training because they are not learned.

The simplest encoding assigns the value `n / K` to the nth position in a sequence of length `K`, with `n` beginning at zero. The `position_encoding_simple` function in `transformers.py` implements this representation. The expected numerical error is below `1e-9`.

### Simple positional encoding

In [ ]:
from transformers_sentiment_analysis import position_encoding_simple

reset_seed(0)
K = 4
M = emb_size = 4

y = position_encoding_simple(K, M)
y_expected = torch.tensor(
    [
        [
            [0.00000, 0.00000, 0.00000, 0.00000],
            [0.25000, 0.25000, 0.25000, 0.25000],
            [0.50000, 0.50000, 0.50000, 0.50000],
            [0.75000, 0.75000, 0.75000, 0.75000],
        ]
    ]
)

print("position_encoding_simple error: ", rel_error(y, y_expected))

K = 5
M = emb_size = 3


y = position_encoding_simple(K, M)
y_expected = torch.tensor(
    [
        [
            [0.00000, 0.00000, 0.00000],
            [0.20000, 0.20000, 0.20000],
            [0.40000, 0.40000, 0.40000],
            [0.60000, 0.60000, 0.60000],
            [0.80000, 0.80000, 0.80000],
        ]
    ]
)
print("position_encoding_simple error: ", rel_error(y, y_expected))

position_encoding_simple error:  0.0
position_encoding_simple error:  0.0


### Sentiment Analysis with the Transformer Model

This section loads and preprocesses sentiment-analysis data using Hugging Face libraries.

- `AutoTokenizer` and `DataCollatorWithPadding` tokenize the data and generate padding masks.
- A PyTorch data loader supplies batches for model training and evaluation.

In [ ]:
! pip install transformers datasets

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-cased')
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-cased', vocab_size=28996, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [ ]:
from datasets import load_dataset
raw_datasets = load_dataset("glue", "sst2")


In [ ]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [ ]:

def tokenize_fn(batch):
  return tokenizer(batch['sentence'], truncation=True)
# map toeknize function to dataset
tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 1821
    })
})

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

In [ ]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1821
    })
})

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    batch_size=8,
    collate_fn=data_collator
)
valid_loader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=8,
    collate_fn=data_collator
)

In [ ]:
# check how it works
for batch in train_loader:
  for k, v in batch.items():
    print("k:", k, "v.shape:", v.shape)
  break


You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


k: labels v.shape: torch.Size([8])
k: input_ids v.shape: torch.Size([8, 26])
k: attention_mask v.shape: torch.Size([8, 26])


In [ ]:
batch.items

<bound method BatchEncoding.items of {'labels': tensor([0, 1, 0, 1, 1, 1, 1, 1]), 'input_ids': tensor([[  101,  1103,  1642,  1110, 24017,   117,  1103, 13948,  1132,  4701,
          5387,  2879, 14550,   117,  1105,  1103,  9688,  1114,   187, 19429,
          1200,  1110, 23609, 18537,   119,   102],
        [  101,  2265,  6608,  5558,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1115,  1122, 17462,  1116,  1315,  1242,  3073,  8057, 27647,
         11603,  1642,  3050,  1154,  1103,  1919,  1159,   102,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1482,   117,   170,  1762,  8124,  6066,  9688,  1111, 13663,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,   170,  2426, 14524, 24836,  1578,   117,   172, 

In [ ]:
from transformers_sentiment_analysis import Transformer_encoder

checkpoint = 'distilbert-base-cased'
num_heads = 2
emb_dim = 512
feedforward_dim = 512
dropout = 0.3
num_enc_layers = 2
vocab_len = 100000
n_classes = 2

model = Transformer_encoder(num_heads, emb_dim, feedforward_dim, dropout, num_enc_layers, vocab_len, n_classes)
model.to('cuda')

Transformer_encoder(
  (emb_layer): Embedding(100000, 512)
  (avg_pool): AdaptiveAvgPool1d(output_size=1)
  (fc): Linear(in_features=512, out_features=2, bias=True)
  (encoder): Encoder(
    (layers): ModuleList(
      (0-1): 2 x EncoderBlock(
        (multihead_attention): MultiHeadAttention(
          (heads): ModuleList(
            (0-1): 2 x SelfAttention(
              (q): Linear(in_features=512, out_features=512, bias=True)
              (k): Linear(in_features=512, out_features=512, bias=True)
              (v): Linear(in_features=512, out_features=512, bias=True)
            )
          )
          (linear): Linear(in_features=1024, out_features=512, bias=True)
        )
        (norm1): LayerNormalization()
        (norm2): LayerNormalization()
        (feedforward): FeedForwardBlock(
          (linear1): Linear(in_features=512, out_features=512, bias=True)
          (linear2): Linear(in_features=512, out_features=512, bias=True)
        )
        (dropout): Dropout(p=0.3, i

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

In [ ]:
import numpy as np
from datetime import datetime
device ='cuda'
# Training loop
def train(model, criterion, optimizer, train_loader, valid_loader, epochs):
  train_losses = np.zeros(epochs)
  test_losses = np.zeros(epochs)

  for it in range(epochs):
    model.train()   # Model in training mode
    t0 = datetime.now()
    train_loss = 0
    n_train = 0
    for batch in train_loader:
      # move data to GPU
      batch = {k: v.to(device) for k, v in batch.items()}

      # zero the parameter gradients
      optimizer.zero_grad()

      # Forward pass
      outputs = model(batch['input_ids'])

      loss = criterion(outputs, batch['labels'])

      # Backward and optimize
      loss.backward()   # Compute Gradients (Back prop)
      optimizer.step()  # Update weights(GD/Adam)

      train_loss += loss.item()*batch['input_ids'].size(0)
      n_train += batch['input_ids'].size(0)

    # Get average train loss
    train_loss = train_loss / n_train

    # Evalaute model at the end of each epoch
    model.eval()
    test_loss = 0
    n_test = 0
    for batch in valid_loader:
      batch = {k: v.to(device) for k, v in batch.items()}
      outputs = model(batch['input_ids'])
      loss = criterion(outputs, batch['labels'])
      test_loss += loss.item()*batch['input_ids'].size(0)
      n_test += batch['input_ids'].size(0)
    test_loss = test_loss / n_test

    # Save losses
    train_losses[it] = train_loss
    test_losses[it] = test_loss

    dt = datetime.now() - t0
    print(f'Epoch {it+1}/{epochs}, Train Loss: {train_loss:.4f}, \
      Test Loss: {test_loss:.4f}, Duration: {dt}')

  return train_losses, test_losses

In [ ]:
train(model, criterion, optimizer, train_loader, valid_loader, epochs=4)

Epoch 1/4, Train Loss: 0.6916,       Test Loss: 0.6962, Duration: 0:03:51.578742
Epoch 2/4, Train Loss: 0.6915,       Test Loss: 0.6970, Duration: 0:03:51.174701
Epoch 3/4, Train Loss: 0.6910,       Test Loss: 0.6939, Duration: 0:03:52.225623
Epoch 4/4, Train Loss: 0.6908,       Test Loss: 0.6959, Duration: 0:03:52.117823


(array([0.69159692, 0.69150912, 0.69096698, 0.69075222]),
 array([0.69616913, 0.69696854, 0.69388584, 0.69587729]))